# 지역별 전력 소비량 예측 — 시계열 EDA와 모델 (PJM Energy)

- 데이터: 미국 동부 송전망 지역별 시간당 전력 (178,262행)
- 목표: 지역별 전력 소비량 예측 — 다중 시계열
- 흐름: 불러오기(wide→long) → 시계열 EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 data_PJM_energy.txt

- 이 데이터의 핵심: **wide→long 변환** · 다중 지역(전역 vs 지역 모델)

## 1. 불러오기 — wide 형식

- 이 파일은 지역이 **컬럼**으로 나열된 wide 형식
- Datetime + 지역 11개 + PJM_Load(초기 전체부하)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("pjm_hourly_est.csv", parse_dates=["Datetime"])
print(df.shape)          # (178262, 13)
print(df.columns.tolist())
df.head()

## 2. 학습 전 확인

- wide 형식을 다중 시계열(long)로 바꿔야 함
- 그 전에 걸러낼 것 확인

### 2-1. PJM_Load 제외 (지역 아님)

- 13번째 PJM_Load는 1998~2002 초기 전체 부하
- 다른 지역과 기간이 안 겹침 → 섞으면 안 됨

In [ ]:
# PJM_Load 관측 기간 확인
pl = df.loc[df["PJM_Load"].notna(), "Datetime"]
print("PJM_Load:", pl.min().date(), "~", pl.max().date())

df = df.drop(columns=["PJM_Load"])   # 제거
print("제거 후 컬럼:", df.columns.tolist())

### 2-2. wide → long 변환 (핵심)

- 지역이 컬럼 → 지역을 item_id 행으로 바꿈 (melt)
- 지역별 관측 안 된 기간은 빈칸(NaN) → 제거

In [ ]:
long = df.melt(id_vars="Datetime", var_name="item_id",
               value_name="target").dropna(subset=["target"])

# 일별 합계 집계 (전력량은 누적 → sum)
long = (long.set_index("Datetime")
            .groupby("item_id")["target"]
            .resample("D").sum().reset_index())
long.columns = ["item_id", "timestamp", "target"]
print(long.shape)
print("지역:", long["item_id"].nunique())
long.head()

## 3. 시계열 EDA

### 3-1. 지역별 관측 기간 다름

- 지역마다 시작·종료가 제각각
- NI는 2011년 종료, EKPC는 2013년 시작

In [ ]:
(long.groupby("item_id")["timestamp"]
   .agg(["min","max","size"]))

### 3-2. 지역 간 스케일 차이 (22배)

- PJME(동부 전체)는 크고, EKPC는 작음

In [ ]:
(long.groupby("item_id")["target"].mean()
   .sort_values(ascending=False)
   .plot(kind="bar", figsize=(9,3), title="mean daily load by region"))
plt.show()
# PJME(최대) vs EKPC(최소) → 약 22배

### 3-3. 계절성 — 여름·겨울 피크

- 전력은 냉난방으로 여름·겨울에 피크

In [ ]:
pjme = long[long["item_id"]=="PJME"].set_index("timestamp")["target"]

pjme.resample("MS").mean().plot(figsize=(12,3),
    title="PJME monthly mean - seasonal peaks")
plt.show()
# 여름(냉방)·겨울(난방) 피크, 봄가을 저점

## 4. 시계열 형식 변환

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts = TimeSeriesDataFrame.from_data_frame(
    long, id_column="item_id", timestamp_column="timestamp")
ts = ts.convert_frequency(freq="D")   # 빈 날짜 채움
print("변환 완료:", ts.shape)

## 5. 학습 — 전역 모델 (여러 지역 동시)

- 여러 지역을 하나의 모델이 함께 학습 (강의 64p)
- prediction_length = 14 (2주)

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

prediction_length = 14
train_data, test_data = ts.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="D",
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)

## 6. 해석

In [ ]:
predictions = predictor.predict(train_data)

predictor.plot(
    data=test_data,
    predictions=predictions,
    item_ids=["PJME"],
    max_history_length=90,
)
plt.show()

### 6-1. 생각해 볼 점

- 여러 지역을 함께 학습(전역)한 것이 유리했는가
- 지역별 패턴이 비슷하면 전역 모델이 서로 도움
- 스케일 22배 차이는 문제가 되지 않았는가
  → 규모 비슷한 지역끼리 묶어 비교해 볼 것

- 다른 다중 시계열과 비교:
  Berkeley(남북반구 반대) · 보행자(스케일 12배)
  → PJM 지역들은 패턴이 유사한 편

## 정리

- wide 형식 → melt로 long 변환 (다중 시계열의 핵심)
- PJM_Load 제외 (지역 아님)
- 집계는 합계(전력량 누적)
- 지역별 기간 다름 → convert_frequency로 채움
- 스케일 22배 · 여름겨울 피크
- 전역 모델로 여러 지역 동시 학습

- 시계열의 다중 학습: 여러 계열이 서로 패턴을 공유
  → 언제 전역이 유리한지 판단하는 것은 사람 몫